### Import Necessary Packages and Libraries

In [18]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import make_scorer, mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

import joblib

In [2]:
# Read in csv
df = pd.read_csv('~/Desktop/ADA/ADA-final-project/data_ingestion/preprocessed_award_data.csv')
print(df.shape)
df_embed = pd.read_csv('contract_embeddings.csv')

(247241, 36)


In [3]:
df = df.merge(df_embed, on = 'generated_internal_id')
print(df.shape)

(247241, 420)


# Predict Total Value of Award (before looking at any modification / follow on information)

In [6]:
# Define columns for Machine Learning Processing
categorical_predictors = ['Awarding Agency' , 'Awarding Sub Agency', 'Funding Agency',
                         'Funding Sub Agency', 'Place of Performance Country Code', 'Place of Performance State Code',
                         'is_democrat', 'naics_code', 'start_year', 'award_not_fund_agency', 'award_not_fund_sub_agency'] # leave out psc_code (~ 900 values)
emb_cols = [col for col in df_embed if col != 'generated_internal_id']
continuous_predictors = ['market_share'] + emb_cols
target = 'Award Amount'
metadata = ['internal_id', 'Award ID_x', 'generated_internal_id', 'Recipient Name', 'recipient_id']

In [7]:
df = df[categorical_predictors + continuous_predictors + [target]]
df = df.dropna() # Only drops a few thousand records - insignificant comparatively

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 242266 entries, 0 to 247240
Columns: 397 entries, Awarding Agency to Award Amount
dtypes: bool(2), float64(388), int64(1), object(6)
memory usage: 732.4+ MB


## Baseline Linear Regression
### Run K Fold Cross Validation Grid Search

In [10]:
# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), continuous_predictors),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_predictors)
    ])

# 2. Update Pipeline to use ElasticNet (Regression)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', ElasticNet(max_iter=5000))
])

# 3. Data Split (Ensure 'y' is continuous, not classes)
X = df[categorical_predictors + continuous_predictors]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define Hyperparameters
# alpha: total penalty strength
# l1_ratio: 1.0 is Lasso, 0.0 is Ridge, anything in between is Elastic Net
param_grid = {
    'regressor__alpha': [0.01, 0.1, 1, 10],
    'regressor__l1_ratio': [0.1, 0.5, 0.7, 0.9]
}

In [11]:
# Grid Search with MAPE scoring
# greater_is_better=False because we want to minimize the error
mape_scorer = make_scorer(mean_absolute_percentage_error, greater_is_better=False)

grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=5, 
    scoring=mape_scorer, 
    n_jobs=-1, 
    verbose=3 
)

print("Starting Grid Search for Elastic Net...")
grid_search.fit(X_train, y_train)

# Evaluation
print(f"\nBest Parameters: {grid_search.best_params_}")
# We multiply by -1 because Scikit-learn negates "loss" scores for maximization
print(f"Best CV MAPE: {-grid_search.best_score_:.4f}")

y_pred = grid_search.predict(X_test)
test_mape = mean_absolute_percentage_error(y_test, y_pred)
print(f"Test Set MAPE: {test_mape:.4f}")

Starting Grid Search for Elastic Net...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV 1/5] END regressor__alpha=0.01, regressor__l1_ratio=0.1;, score=-87093310045886644224.000 total time=20.9min
[CV 4/5] END regressor__alpha=0.01, regressor__l1_ratio=0.9;, score=-81765440516159258624.000 total time=40.0min
[CV 5/5] END regressor__alpha=0.01, regressor__l1_ratio=0.9;, score=-86028060494292729856.000 total time=21.5min


/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV 3/5] END regressor__alpha=0.01, regressor__l1_ratio=0.1;, score=-81787702478514847744.000 total time=115.3min
[CV 2/5] END regressor__alpha=0.1, regressor__l1_ratio=0.5;, score=-80990383415434723328.000 total time= 4.8min
[CV 1/5] END regressor__alpha=0.01, regressor__l1_ratio=0.7;, score=-90389444036876517376.000 total time=139.4min
[CV 5/5] END regressor__alpha=0.1, regressor__l1_ratio=0.7;, score=-82956638337029832704.000 total time=74.4min
[CV 3/5] END regressor__alpha=1, regressor__l1_ratio=0.7;, score=-74526067541955067904.000 total time=68.8min
[CV 2/5] END regressor__alpha=10, regressor__l1_ratio=0.7;, score=-65992816646098370560.000 total time=34.0min
[CV 3/5] END regressor__alpha=0.01, regressor__l1_ratio=0.7;, score=-81983391443455967232.000 total time=155.7min
[CV 4/5] END regressor__alpha=0.1, regressor__l1_ratio=0.9;, score=-79959755483878572032.000 total time=87.2min
[CV 2/5] END regressor__alpha=10, regressor__l1_ratio=0.1;, score=-63151258847950741504.000 total tim

## Decision Tree

In [13]:
# Define the Decision Tree Pipeline
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DecisionTreeRegressor(random_state=42))
])

# Hyperparameters: focus on depth and leaf constraints to prevent overfitting
dt_params = {'regressor__max_depth': [5, 10, 20, None], 
             'regressor__min_samples_split': [2, 10]}

dt_grid = GridSearchCV(dt_pipeline, dt_params, cv=5, scoring=mape_scorer, n_jobs=-1, verbose = 3)
dt_grid.fit(X_train, y_train)

print(f"\nBest Parameters: {dt_grid.best_params_}")
# We multiply by -1 because Scikit-learn negates "loss" scores for maximization
print(f"Best CV MAPE: {-dt_grid.best_score_:.4f}")

y_pred = dt_grid.predict(X_test)
test_mape = mean_absolute_percentage_error(y_test, y_pred)
print(f"Test Set MAPE: {test_mape:.4f}")

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV 4/5] END regressor__alpha=0.01, regressor__l1_ratio=0.7;, score=-81355842657422458880.000 total time=177.8min
[CV 3/5] END regressor__alpha=1, regressor__l1_ratio=0.1;, score=-70032687544104050688.000 total time=41.2min
[CV 5/5] END regressor__alpha=1, regressor__l1_ratio=0.7;, score=-75940963334022627328.000 total time=68.4min
[CV 3/5] END regressor__alpha=10, regressor__l1_ratio=0.7;, score=-63666285122634932224.000 total time=29.2min
[CV 4/5] END regressor__max_depth=5, regressor__min_samples_split=2;, score=-50822786612520198144.000 total time= 1.9min


/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV 3/5] END regressor__max_depth=10, regressor__min_samples_split=2;, score=-51747069927775666176.000 total time= 4.1min
[CV 5/5] END regressor__max_depth=20, regressor__min_samples_split=2;, score=-71501566798091681792.000 total time= 8.0min
[CV 4/5] END regressor__max_depth=10, regressor__min_samples_split=2;, score=-49228262169269616640.000 total time= 4.1min
[CV 2/5] END regressor__max_depth=20, regressor__min_samples_split=10;, score=-54534773757755899904.000 total time= 8.0min
[CV 5/5] END regressor__max_depth=10, regressor__min_samples_split=2;, score=-53927727827283533824.000 total time= 4.0min
[CV 4/5] END regressor__max_depth=20, regressor__min_samples_split=10;, score=-48547413663689023488.000 total time= 7.5min
[CV 1/5] END regressor__alpha=0.1, regressor__l1_ratio=0.1;, score=-80732104432288153600.000 total time= 9.4min
[CV 2/5] END regressor__alpha=0.1, regressor__l1_ratio=0.1;, score=-79646482187527880704.000 total time= 6.1min
[CV 3/5] END regressor__alpha=0.1, regress

## Random Forest

In [ ]:
# Define the Random Forest Pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1))
])

# Hyperparameters: focus on depth and leaf constraints to prevent overfitting
rf_params = {'regressor__n_estimators': [100, 200], 
             'regressor__max_depth': [10, 20]}

rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=5, scoring=mape_scorer, n_jobs=-1, verbose = 3)
rf_grid.fit(X_train, y_train)

print(f"\nBest Parameters: {rf_grid.best_params_}")
# We multiply by -1 because Scikit-learn negates "loss" scores for maximization
print(f"Best CV MAPE: {-rf_grid.best_score_:.4f}")

y_pred = rf_grid.predict(X_test)
test_mape = mean_absolute_percentage_error(y_test, y_pred)
print(f"Test Set MAPE: {test_mape:.4f}")

## XGBoost

In [ ]:
# Define the XGBoost Pipeline
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=42, objective='reg:squarederror'))
])

# Hyperparameters: focus on depth and leaf constraints to prevent overfitting
xgb_params = {'regressor__learning_rate': [0.01, 0.1], 
              'regressor__n_estimators': [100, 300], 
              'regressor__max_depth': [3, 6]}

xgb_grid = GridSearchCV(xgb_pipeline, xgb_params, cv=5, scoring=mape_scorer, n_jobs=-1, verbose = 3)
xgb_grid.fit(X_train, y_train)

print(f"\nBest Parameters: {xgb_grid.best_params_}")
# We multiply by -1 because Scikit-learn negates "loss" scores for maximization
print(f"Best CV MAPE: {-xgb_grid.best_score_:.4f}")

y_pred = xgb_grid.predict(X_test)
test_mape = mean_absolute_percentage_error(y_test, y_pred)
print(f"Test Set MAPE: {test_mape:.4f}")

# Save Models and Metrics to Disk

In [15]:
version = "v1"

# IMPORTANT!!!!
## Ensure Proper Version Number above is specified before running!

In [20]:
# Dictionary mapping names to your trained GridSearchCV objects
model_dict = {
    "Elastic Net": grid_search,
    "Decision Tree": dt_grid,
    # "Random Forest": rf_grid,
    # "XGBoost": xgb_grid
}

results_list = []

for name, grid in model_dict.items():
    # Save the best estimator
    clean_name = name.lower().replace(' ', '_')
    filename = f"models/regression/{clean_name}_model_regression_{version}.joblib"
    joblib.dump(grid.best_estimator_, filename)
    
    # 2. Get predictions
    y_pred = grid.predict(X_test)
    
    # 3. Calculate Regression Metrics
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # 4. Append to results
    results_list.append({
        "model_name": f"{name}_reg_{version}",
        "model_location": filename,
        "MAPE": round(mape, 4),
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2_Score": round(r2, 4)
    })

# Create/Update the summary CSV
df_metrics = pd.DataFrame(results_list)

# try:
#     df_metrics_old = pd.read_csv("amount_regression_model_evaluation_metrics.csv")
#     df_metrics = pd.concat([df_metrics_old, df_metrics], axis=0).drop_duplicates(subset=['model_name'])
# except FileNotFoundError:
#     pass

df_metrics.to_csv("amount_regression_model_evaluation_metrics.csv", index=False)

print("Regression models saved and metrics CSV generated successfully.")
display(df_metrics)

Regression models saved and metrics CSV generated successfully.


,model_name,model_location,MAPE,MAE,RMSE,R2_Score
0,Elastic Net_reg_v1,models/regression/elastic_net_model_regression...,6.179195e+19,1285601.83,6293819.84,0.0204
1,Decision Tree_reg_v1,models/regression/decision_tree_model_regressi...,7.306905e+19,1361773.57,10273197.74,-1.6098
